In [1]:
import pandas as pd

In [2]:
input_path = "../../data/converted/Online_Retail.csv"
output_path = "../../data/processed/Online_Retail.csv"

df = pd.read_csv(input_path)
print("Raw shape: ", df.shape)
df.head()

Raw shape:  (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.info()
df.describe()
df.isnull().sum()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [4]:
df = df.drop_duplicates()

In [5]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

### FLAG

In [6]:

df["is_cancelled"] = df["InvoiceNo"].astype(str).str.startswith("C")
df["is_return"] = df["Quantity"] < 0
df["is_missing_customer"] = df["CustomerID"].isnull()
df["is_invalid_price"] = df["UnitPrice"] <= 0

q_low = df["Quantity"].quantile(0.01)
q_high = df["Quantity"].quantile(0.99)
df["is_outlier_quantity"] = (df["Quantity"] < q_low) | (df["Quantity"] > q_high)

In [7]:
df["Revenue"] = (df["Quantity"] * df["UnitPrice"]).round(2)

df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")
df["Hour"] = df["InvoiceDate"].dt.hour
df["Weekday"] = df["InvoiceDate"].dt.day_name()

In [8]:
df_clean = df[
    (~df["is_cancelled"]) &
    (~df["is_return"]) &
    (~df["is_missing_customer"]) &
    (~df["is_invalid_price"])
].copy()

# Drop flag columns
flag_cols = [col for col in df.columns if col.startswith("is_")]
df_clean = df_clean.drop(columns=flag_cols)

print("Clean shape:", df_clean.shape)

Clean shape: (392692, 12)


In [9]:
df.to_csv("../../data/processed/Online_Retail_flagged.csv", index=False)
df_clean.to_csv("../../data/processed/Online_Retail_clean.csv", index=False)

In [10]:
df_flagged = pd.read_csv("../../data/processed/Online_Retail_flagged.csv")

print(df_flagged["is_cancelled"].value_counts())
print(df_flagged["is_return"].value_counts())
print(df_flagged["is_missing_customer"].value_counts())

is_cancelled
False    527390
True       9251
Name: count, dtype: int64
is_return
False    526054
True      10587
Name: count, dtype: int64
is_missing_customer
False    401604
True     135037
Name: count, dtype: int64
